# YOLO11n-Seg Training on Synthetic Silhouettes

Finetune YOLO11n-seg on the synthetic silhouette dataset (single class: `person`).

**Prerequisite:** Generate the dataset locally or with notebook 03 using `--mode both` (or `--mode mask`). Upload the resulting `synthetic-seg` folder to Drive.

**Setup:** Runtime > Change runtime type > **GPU (T4)** or **TPU**

## 1. Install Dependencies

In [ ]:
!pip install ultralytics -q

## 2. Check Hardware

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Check for TPU
try:
    import torch_xla.core.xla_model as xm
    print(f"TPU available: {xm.xla_device()}")
except ImportError:
    print("TPU: not available (GPU runtime selected)")

## 3. Mount Google Drive

Upload your `synthetic-seg` folder to Google Drive before running this cell.

Expected structure on Drive:
```
My Drive/
  PointsX/
    synthetic-seg/
      train/
        images/   (white-on-black silhouette .jpg)
        labels/   (YOLO polygon .txt: `0 x1 y1 x2 y2 ...`)
      val/
        images/
        labels/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURE: Set your Google Drive path here
# ============================================================
DRIVE_DATASET_DIR = "/content/drive/MyDrive/PointsX/synthetic-seg"

# Verify dataset exists
import os
for split in ["train", "val"]:
    imgs = os.path.join(DRIVE_DATASET_DIR, split, "images")
    lbls = os.path.join(DRIVE_DATASET_DIR, split, "labels")
    n_imgs = len(os.listdir(imgs)) if os.path.isdir(imgs) else 0
    n_lbls = len(os.listdir(lbls)) if os.path.isdir(lbls) else 0
    print(f"{split}: {n_imgs} images, {n_lbls} labels")

## 4. Copy Dataset to Local Storage (Faster I/O)

Training directly from Google Drive is slow. Copy to Colab's local SSD first.

In [ ]:
LOCAL_DATASET_DIR = "/content/synthetic-seg"

if not os.path.exists(LOCAL_DATASET_DIR):
    print("Copying dataset to local storage (this takes a few minutes)...")
    !cp -r "{DRIVE_DATASET_DIR}" "{LOCAL_DATASET_DIR}"
    print("Done!")
else:
    print("Dataset already copied locally.")

# Verify
for split in ["train", "val"]:
    n = len(os.listdir(f"{LOCAL_DATASET_DIR}/{split}/images"))
    print(f"{split}: {n} images")

## 5. (Optional) Regenerate YOLO Polygon Labels

If your `labels/` folder is missing or you re-rendered masks, convert silhouettes → YOLO polygons with OpenCV. Skip this cell if labels already exist.

In [ ]:
!pip install opencv-python rich -q

# Use the repo's masks_to_polygons script — fetch it straight from GitHub to avoid a full repo clone
import urllib.request
url = "https://raw.githubusercontent.com/feed7362/PointsX/master/src/pointsx/synthetic/masks_to_polygons.py"
urllib.request.urlretrieve(url, "/content/masks_to_polygons.py")

# Only run if train/labels is empty
if not os.listdir(f"{LOCAL_DATASET_DIR}/train/labels"):
    !python /content/masks_to_polygons.py --data {LOCAL_DATASET_DIR}
else:
    print("Labels already present — skipping conversion.")

## 6. Create Dataset YAML

In [ ]:
DATASET_YAML = "/content/synthetic-seg.yaml"

yaml_content = f"""# Synthetic silhouette dataset - 1-class body segmentation
path: {LOCAL_DATASET_DIR}
train: train/images
val: val/images

names:
  0: person
"""

with open(DATASET_YAML, "w") as f:
    f.write(yaml_content)

print(f"Dataset YAML written to: {DATASET_YAML}")
print(yaml_content)

## 7. Sanity-Check Labels

Overlay a few polygons on their mask images to confirm everything lines up.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

img_dir   = Path(LOCAL_DATASET_DIR) / "train" / "images"
label_dir = Path(LOCAL_DATASET_DIR) / "train" / "labels"

samples = random.sample(sorted(img_dir.glob("*.jpg")), min(4, len(list(img_dir.glob("*.jpg")))))
fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
if len(samples) == 1:
    axes = [axes]

for ax, img_path in zip(axes, samples):
    img = np.array(Image.open(img_path))
    ax.imshow(img, cmap="gray")
    lbl = label_dir / img_path.with_suffix(".txt").name
    if lbl.exists():
        parts = lbl.read_text().split()
        coords = np.array(parts[1:], dtype=float).reshape(-1, 2)
        H, W = img.shape[:2]
        xs = coords[:, 0] * W
        ys = coords[:, 1] * H
        ax.plot(np.append(xs, xs[0]), np.append(ys, ys[0]), color="lime", linewidth=1.5)
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Train YOLO11n-Seg

In [ ]:
from ultralytics import YOLO

# ============================================================
# CONFIGURE: Training hyperparameters
# ============================================================
EPOCHS     = 50         # seg usually converges faster than pose; lower if you see early plateau
BATCH_SIZE = -1         # auto (maximizes GPU memory usage)
IMG_SIZE   = 640

model = YOLO("yolo11n-seg.pt")  # downloads automatically

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project="/content/runs",
    name="seg",
    exist_ok=True,
    pretrained=True,
    patience=15,
    workers=2,
    device=0,  # GPU 0
)

## 9. View Results

In [ ]:
from IPython.display import Image, display

# Training curves
display(Image(filename="/content/runs/seg/results.png", width=800))

In [ ]:
# Validation predictions (mask overlays)
display(Image(filename="/content/runs/seg/val_batch0_pred.jpg", width=800))

In [ ]:
# Confusion matrix
display(Image(filename="/content/runs/seg/confusion_matrix.png", width=600))

## 10. Validate Best Model

In [ ]:
best_model = YOLO("/content/runs/seg/weights/best.pt")
metrics = best_model.val(data=DATASET_YAML)

print(f"\nBox  mAP50:    {metrics.box.map50:.4f}")
print(f"Box  mAP50-95: {metrics.box.map:.4f}")
print(f"Seg  mAP50:    {metrics.seg.map50:.4f}")
print(f"Seg  mAP50-95: {metrics.seg.map:.4f}")

## 11. Test Inference

In [ ]:
import glob

# Run inference on a few validation images
val_images = sorted(glob.glob(f"{LOCAL_DATASET_DIR}/val/images/*.jpg"))[:4]
results = best_model(val_images, imgsz=640)

for r in results:
    im = r.plot()
    display(Image(data=r.save(filename=None)))

## 12. Save Best Model to Google Drive

In [ ]:
import shutil

DRIVE_OUTPUT = "/content/drive/MyDrive/PointsX/runs/seg"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy best and last weights
for name in ["best.pt", "last.pt"]:
    src = f"/content/runs/seg/weights/{name}"
    dst = f"{DRIVE_OUTPUT}/{name}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Saved: {dst}")

# Copy results
for name in ["results.csv", "results.png"]:
    src = f"/content/runs/seg/{name}"
    dst = f"{DRIVE_OUTPUT}/{name}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Saved: {dst}")

print("\nAll results saved to Google Drive!")